In [0]:
%sql
-- Step 1: Define a SQL function in Unity Catalog that computes a bonus.
-- The bonus is calculated as 10% of the given salary.
-- Stored in the hr_catalog.hr_core schema so it can be reused across notebooks and tools.
CREATE OR REPLACE FUNCTION hr_catalog.hr_core.calculate_bonus(
    salary DOUBLE
)
RETURNS DOUBLE
RETURN salary * 0.10;

In [0]:
%sql
-- Step 2: Quick sanity check — call the function directly with a sample salary of 100000.
-- Expected result: 10000.0 (10% of 100000).
SELECT hr_catalog.hr_core.calculate_bonus(100000);

hr_catalog.hr_core.calculate_bonus(100000)
10000.0


In [0]:
# Step 3: Wrap the SQL function in a Python "tool" callable.
# This function uses the Databricks SDK (WorkspaceClient) to submit a SQL statement
# to a SQL warehouse and return the scalar result. It acts as the bridge between
# the agent (Python) and the underlying SQL business logic.
from databricks.sdk import WorkspaceClient

# Initialize a workspace client using the current notebook's authentication.
w = WorkspaceClient()

# The SQL warehouse ID used to execute statements.
WAREHOUSE_ID = "1cfe7f2931b647ba"

def calculate_bonus_tool(salary):
    """Call the Unity Catalog SQL function calculate_bonus and return the bonus value."""
    # Submit the SQL statement to the warehouse.
    response = w.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID,
        statement=f"""
        SELECT hr_catalog.hr_core.calculate_bonus({salary})
        """,
        wait_timeout="30s"
    )

    # Parse the JSON response and extract the scalar result from the first row.
    result = response.as_dict()

    return result["result"]["data_array"][0][0]

In [0]:
# Step 4: Verify the Python tool wrapper works correctly.
# Calls the tool with a salary of 100000 and should return 10000.0.
calculate_bonus_tool(100000)

'10000.0'

In [0]:
# Step 5: Build a simple HR agent that decides whether a tool call is needed.
# The agent inspects the user's question using keyword matching. If the question
# mentions "bonus" and contains a numeric salary, it routes to calculate_bonus_tool.
# Otherwise, it responds with a default message indicating no tool was required.
import re

def hr_agent(question):
    """Route an HR-related question to the appropriate tool or return a fallback answer."""
    question_lower = question.lower()

    # Check if the question is about bonuses.
    if "bonus" in question_lower:

        # Extract the first integer from the question (assumed to be the salary).
        numbers = re.findall(r"\d+", question)

        if numbers:
            salary = int(numbers[0])
            # Delegate to the tool to compute the actual bonus.
            bonus = calculate_bonus_tool(salary)

            return (
                f"The bonus for a salary of "
                f"{salary} is {bonus}."
            )

    # Fallback: no matching tool was found for this question.
    return (
        "I can help answer general HR questions. "
        "No tool call was required."
    )

In [0]:
# Step 6: Test the agent with a bonus-related question.
# Expected: the agent detects "bonus" and the salary 100000, calls the tool,
# and returns "The bonus for a salary of 100000 is 10000.0."
hr_agent(
    "What is the bonus for a salary of 100000?"
)

'The bonus for a salary of 100000 is 10000.0.'

In [0]:
# Step 7: Test the agent with a question that does NOT require a tool call.
# Expected: the agent finds no matching keyword/tool and returns the fallback message.
hr_agent(
    "What is GDPR?"
)

'I can help answer general HR questions. No tool call was required.'